# GDSC EDA
Preliminary exploration of the four GDSC source files used in the GeneTraceAI pipeline.

Files covered:
- `model_list_20260709.csv` — cell-line metadata (2 266 models, 98 cols)
- `screened_compounds_rel_8.5.csv` — drug catalogue (621 drugs)
- `Project_Score_piority_scores_Sanger_v2_Broad_21Q2_20240111.csv` — priority scores per gene/cancer (713 rows)
- `GDSC2_fitted_dose_response_27Oct23.xlsx` — fitted dose-response (242 036 rows)

In [ ]:
import pandas as pd
import os

DATA = os.path.join('..', '..', 'data', 'GDSC')

model_list       = pd.read_csv(os.path.join(DATA, 'model_list_20260709.csv'), low_memory=False)
compounds        = pd.read_csv(os.path.join(DATA, 'screened_compounds_rel_8.5.csv'))
project_score    = pd.read_csv(os.path.join(DATA, 'Project_Score_piority_scores_Sanger_v2_Broad_21Q2_20240111.csv'))
dose_response    = pd.read_excel(os.path.join(DATA, 'GDSC2_fitted_dose_response_27Oct23.xlsx'))

print('All files loaded.')

## 1. Shape & Schema Report

In [ ]:
frames = {
    'model_list':    model_list,
    'compounds':     compounds,
    'project_score': project_score,
    'dose_response': dose_response,
}

for name, df in frames.items():
    print(f'\n=== {name} ===')
    print(f'  rows × cols : {df.shape}')
    print(f'  columns     : {list(df.columns)}')
    print(f'  dtypes      :\n{df.dtypes.to_string()}')
    null_pct = (df.isnull().mean() * 100).round(1)
    high_null = null_pct[null_pct > 20]
    if not high_null.empty:
        print(f'  cols >20 % null:\n{high_null.to_string()}')

## 2. ID Format Inventory
Check the identifier columns that will be used as join keys across files.

In [ ]:
id_cols = {
    'model_list':    ['model_id', 'SANGER_MODEL_ID', 'COSMIC_ID', 'BROAD_ID', 'CCLE_ID', 'RRID', 'model_name'],
    'dose_response': ['SANGER_MODEL_ID', 'CELL_LINE_NAME', 'DRUG_ID', 'DRUG_NAME'],
    'compounds':     ['DRUG_ID', 'DRUG_NAME'],
    'project_score': ['gene id', 'symbol', 'analysis name'],
}

for name, cols in id_cols.items():
    df = frames[name]
    print(f'\n--- {name} ---')
    for col in cols:
        if col in df.columns:
            sample = df[col].dropna().unique()[:3].tolist()
            print(f'  {col:30s}  nunique={df[col].nunique():6d}  sample={sample}')

## 3. model_list Deep-dive

In [ ]:
# model_id prefix distribution (SIDM vs other)
print('model_id prefix counts:')
print(model_list['model_id'].str[:4].value_counts())

print('\ncancer_type top 15:')
print(model_list['cancer_type'].value_counts().head(15))

print('\ntissue top 15:')
print(model_list['tissue'].value_counts().head(15))

In [ ]:
# Cross-reference ID coverage
for col in ['COSMIC_ID', 'BROAD_ID', 'CCLE_ID', 'RRID']:
    filled = model_list[col].notna().sum()
    print(f'  {col:12s}: {filled}/{len(model_list)} filled ({100*filled/len(model_list):.1f} %)')

## 4. screened_compounds Deep-dive

In [ ]:
print('DRUG_ID range:', compounds['DRUG_ID'].min(), '–', compounds['DRUG_ID'].max())
print('\nTop target pathways:')
print(compounds['TARGET_PATHWAY'].value_counts().head(10))
print('\nScreening sites:')
print(compounds['SCREENING_SITE'].value_counts())

## 5. project_score Deep-dive

In [ ]:
print('Unique cancer types (analysis name):')
print(sorted(project_score['analysis name'].unique()))

print('\nTractability buckets:')
print(project_score['tractability bucket'].value_counts())

print('\nScore distribution:')
print(project_score['score'].describe())

## 6. dose_response Deep-dive

In [ ]:
print('DRUG_ID range:', dose_response['DRUG_ID'].min(), '–', dose_response['DRUG_ID'].max())
print('SANGER_MODEL_ID sample:', dose_response['SANGER_MODEL_ID'].dropna().unique()[:5].tolist())

print('\nLN_IC50 distribution:')
print(dose_response['LN_IC50'].describe())

print('\nAUC distribution:')
print(dose_response['AUC'].describe())

print('\nZ_SCORE distribution:')
print(dose_response['Z_SCORE'].describe())

print('\nTop cancer types by row count:')
print(dose_response['CANCER_TYPE'].value_counts().head(10))

print('\nDrug-cell line coverage (rows per drug, top 10):')
print(dose_response.groupby('DRUG_NAME').size().sort_values(ascending=False).head(10))

## 7. Join Sanity Checks
Check that the SANGER_MODEL_ID in dose_response maps into model_list.

In [ ]:
dr_ids  = set(dose_response['SANGER_MODEL_ID'].dropna())
ml_ids  = set(model_list['model_id'].dropna())

in_both = dr_ids & ml_ids
dr_only = dr_ids - ml_ids
ml_only = ml_ids - dr_ids

print(f'SANGER_MODEL_ID in dose_response   : {len(dr_ids)}')
print(f'model_id in model_list             : {len(ml_ids)}')
print(f'Matched (in both)                  : {len(in_both)}')
print(f'In dose_response only (unmatched)  : {len(dr_only)}')
print(f'In model_list only (not screened)  : {len(ml_only)}')

In [ ]:
# DRUG_ID overlap between dose_response and compounds catalogue
dr_drugs  = set(dose_response['DRUG_ID'].dropna().astype(int))
cat_drugs = set(compounds['DRUG_ID'].dropna().astype(int))

print(f'DRUG_IDs in dose_response   : {len(dr_drugs)}')
print(f'DRUG_IDs in compounds       : {len(cat_drugs)}')
print(f'Matched                     : {len(dr_drugs & cat_drugs)}')
print(f'In dose_response only       : {len(dr_drugs - cat_drugs)}')

## 8. Project Score Fitness TSV Files (CRISPR Screen Matrix)

Four wide-format matrices: rows = genes (17 645), columns = cell lines (1 107).  
Header is **4 rows deep** before data begins:

| Row | Content |
|-----|---------|
| 0 | `model_name` — human-readable cell-line name |
| 1 | `model_id` — SIDM identifier |
| 2 | `source` — Broad or Sanger |
| 3 | `qc_pass` — always TRUE in this release |
| 4 | column labels: `gene_id`, `symbol`, `ensembl_gene_id`, then cell-line names |

Gene index columns (0–2): `gene_id` (SIDG…), `symbol`, `ensembl_gene_id`.  
Data values differ by file — see cell below.

In [ ]:
TSV_FILES = {
    'bayesian_factors':        'fitness_scores_bayesian_factors_20250624.tsv',
    'binary_matrix':           'fitness_scores_binary_matrix_20250624.tsv',
    'fold_change_values':      'fitness_scores_fold_change_values_20250624.tsv',
    'scaled_bayesian_factors': 'fitness_scores_scaled_bayesian_factors_20250624.tsv',
}
TSV_PREFIX = 'Project_score_combined_Sanger_v2_Broad_21Q2_'

tsv_frames = {}

for name, suffix in TSV_FILES.items():
    path = os.path.join(DATA, TSV_PREFIX + suffix)
    raw = pd.read_csv(path, sep='\t', header=None, low_memory=False)

    # Extract metadata from header rows
    model_ids = raw.iloc[1, 3:].dropna().tolist()
    sources   = raw.iloc[2, 3:].value_counts().to_dict()
    qc_pass   = raw.iloc[3, 3:].value_counts().to_dict()

    # Data block: rows 5+, cols 3+ (numeric values)
    data_block = raw.iloc[5:, 3:].apply(pd.to_numeric, errors='coerce')

    gene_ids = raw.iloc[5:, 0].tolist()
    symbols  = raw.iloc[5:, 1].tolist()

    tsv_frames[name] = {
        'raw': raw, 'data': data_block,
        'gene_ids': gene_ids, 'model_ids': model_ids,
    }

    print(f'\n=== {name} ===')
    print(f'  Genes (rows)      : {len(gene_ids):,}')
    print(f'  Cell lines (cols) : {len(model_ids):,}')
    print(f'  Sources           : {sources}')
    print(f'  QC pass           : {qc_pass}')
    print(f'  gene_id format    : {gene_ids[:3]}')
    print(f'  gene symbols      : {symbols[:3]}')
    print(f'  SIDM sample       : {model_ids[:3]}')
    print(f'  Value range       : {data_block.stack().min():.4f}  to  {data_block.stack().max():.4f}')
    print(f'  Null %            : {data_block.isnull().mean().mean()*100:.1f} %')

In [ ]:
# TSV → model_list join check (SIDM IDs)
tsv_sidm = set(tsv_frames['bayesian_factors']['model_ids'])
ml_sidm  = set(model_list['model_id'].dropna())

print(f'SIDM IDs in TSV files       : {len(tsv_sidm):,}')
print(f'SIDM IDs in model_list      : {len(ml_sidm):,}')
print(f'Matched                     : {len(tsv_sidm & ml_sidm):,}')
print(f'In TSV only (unmatched)     : {len(tsv_sidm - ml_sidm):,}')

# TSV gene_id → project_score join check
tsv_genes  = set(tsv_frames['bayesian_factors']['gene_ids'])
ps_genes   = set(project_score['gene id'].dropna())
print(f'\nSIDG gene IDs in TSV        : {len(tsv_genes):,}')
print(f'SIDG gene IDs in project_score : {len(ps_genes):,}')
print(f'Overlap                     : {len(tsv_genes & ps_genes):,}')